In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib

In [2]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"

In [3]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [4]:
louisiana

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns
0,0,louisiana,0,0,367983.8318,68239.85746,80.234375,79198.54166,6714.566582,1154589.900,...,9.980000,0.322352,2.821843,4.032755,0.0,39.360000,15.368289,106.770000,4.290000,32.920000
1,0,louisiana,1,0,362385.6368,67201.71394,79.013757,77993.68198,6612.416842,1137024.945,...,9.968923,0.319496,2.800291,4.092947,0.0,40.166298,14.997723,108.605133,3.731796,31.374749
2,0,louisiana,2,0,361096.1267,66962.58391,78.732596,77716.14990,6588.887271,1132978.965,...,9.489254,0.320059,2.788854,4.099808,0.0,38.639666,13.763035,80.719306,3.559992,31.843131
3,0,louisiana,3,0,359935.7476,66747.40026,78.479589,77466.40976,6567.713942,1129338.147,...,10.279229,0.321689,2.777719,4.162110,0.0,41.465412,14.851001,146.300301,3.841408,33.650089
4,0,louisiana,4,0,358773.5212,66531.87404,78.226180,77216.27204,6546.506906,1125691.534,...,9.983308,0.323218,2.772568,4.173150,0.0,37.807488,13.488061,146.740554,3.879013,34.227053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,127127,louisiana,31,0,311979.6878,53280.76885,83.099337,74341.21831,6871.576034,1116082.754,...,2.290044,0.099194,1.610628,2.566150,0.0,9.504129,1.357871,6.354095,0.494434,17.846785
2408,127127,louisiana,32,0,310670.5663,53049.87938,83.075690,74040.77745,6869.511396,1114713.368,...,2.285685,0.087899,1.527726,2.529939,0.0,9.783970,1.280190,4.725632,0.364936,17.059667
2409,127127,louisiana,33,0,309356.4650,52821.26235,83.037733,73734.23638,6866.309713,1113193.917,...,2.281655,0.076939,1.445208,2.497102,0.0,9.535228,1.030977,3.223424,0.239272,16.254621
2410,127127,louisiana,34,0,308044.9743,52594.98521,82.990728,73425.33726,6862.387927,1111581.984,...,2.277804,0.066352,1.363483,2.466951,0.0,9.696216,0.918047,1.841394,0.117569,15.427843


In [5]:
# 1) Filter out the base case
base_case = louisiana[louisiana['primary_id'] == 0].copy()


In [6]:
keywords = ['demand', 'trns']
[col for col in louisiana.columns if all(k in col for k in keywords)]

['energy_demand_enfu_subsector_total_pj_trns_fuel_ammonia',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_biofuels',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_biogas',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_biomass',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_coal',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_coke',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_crude',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_diesel',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_electricity',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_furnace_gas',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_gasoline',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_geothermal',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_hydrocarbon_gas_liquids',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_hydrogen',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_kerosene',
 'energy_demand_enfu_subsector_total_pj_trns_fuel_natural_gas',
 'energy_demand

In [7]:
[col for col in louisiana.columns if "vehicle_distance_traveled_trns_road_light_electricity" in col]

['vehicle_distance_traveled_trns_road_light_electricity']

In [9]:
[col for col in louisiana.columns if startswith(k in col for k in keywords)]

NameError: name 'startswith' is not defined

In [10]:
# 2) Define your fuels
relevant_fuels = [
    'biomass', 'coal', 'coke', 'diesel', 'electricity',
    'furnace_gas', 'gasoline', 'hydrocarbon_gas_liquids',
    'hydrogen', 'kerosene', 'natural_gas', 'oil'
]


In [11]:
# 3) Initialize accumulators
total_fuel_consumption_avoided_by_efficiency = pd.Series(0.0, index=base_case.index)
total_fuel_demand = pd.DataFrame({'time_period': base_case['time_period']}, index=base_case.index)

In [12]:
# 4) Loop over fuels 
for fuel in relevant_fuels:
    # find the efficiency column(s) for this fuel
    eff_cols = [c for c in base_case.columns
                if c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{fuel}')]
    # find the demand column(s) for this fuel
    dem_cols = [c for c in base_case.columns
                if (fuel in c and 'energy_demand_enfu_subsector_total_pj_inen' in c)]
    if not eff_cols or not dem_cols:
        continue

    fuel_efficiency = base_case[eff_cols[0]]
    fuel_demand     = base_case[dem_cols[0]]

    # store demand
    total_fuel_demand[fuel] = fuel_demand

    # calculate avoided consumption
    fuel_consumed          = fuel_demand / fuel_efficiency
    baseline_efficiency    = fuel_efficiency.iloc[0]
    fuel_consumed_baseline = fuel_demand / baseline_efficiency
    delta_consumption      = fuel_consumed_baseline - fuel_consumed

    total_fuel_consumption_avoided_by_efficiency += delta_consumption

In [13]:
# 5) Build the output DataFrame
output_data = pd.DataFrame({
    'time_period': base_case['time_period'],
    'efficiency_energy_saving_in_PJ': total_fuel_consumption_avoided_by_efficiency
}, index=base_case.index)

In [14]:
# 6) Apply your CAPEX/OPEX multipliers
capex_multiplier = 10_000_000
output_data['efficiency_capex'] = output_data['efficiency_energy_saving_in_PJ'] * capex_multiplier
output_data['efficiency_opex']  = 0

In [15]:
# 7) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5

In [16]:

# 8) Demand & cost breakdown
output_data['industrial_energy_demand_for_electricity_in_PJ'] = total_fuel_demand['electricity']
output_data['industrial_energy_demand_for_other_fuels_in_PJ'] = (
    total_fuel_demand[relevant_fuels].sum(axis=1) - total_fuel_demand['electricity']
)

output_data['electricity_capex'] = (
    capex_industrial_electricity * output_data['industrial_energy_demand_for_electricity_in_PJ']
)
output_data['other_fuel_capex'] = (
    capex_industrial_other * output_data['industrial_energy_demand_for_other_fuels_in_PJ']
)
output_data['electricity_opex'] = (
    opex_industrial_electricity * output_data['industrial_energy_demand_for_electricity_in_PJ']
)
output_data['other_fuel_opex'] = (
    opex_industrial_other * output_data['industrial_energy_demand_for_other_fuels_in_PJ']
)

In [17]:


# 9) Write out to CSV

output_data.to_csv(OUTPUT_DIR/'industrial_energy_cost.csv', index=False)
